# Simple: Add/Modify Fields with PAMOLA.CORE

**Goal**: Learn field manipulation basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Add new fields using lookups and conditions
- Modify existing fields based on expressions
- Compare before/after results
- Understand data transformation capabilities

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 01_add_modify_fields_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
import math
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.field_ops.add_modify_fields import (
        AddOrModifyFieldsOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records suitable for field operations

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with various fields for transformation
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'product_code': ['P001', 'P002', 'P003', 'P001', 'P004', 'P002', 'P005', 'P003', 'P001', 'P004',
                        'P005', 'P002', 'P001', 'P003', 'P004', 'P005', 'P001', 'P002', 'P003', 'P004'],
        'quantity': [10, 5, 15, 8, 12, 20, 7, 9, 11, 14, 6, 13, 16, 10, 8, 12, 15, 9, 11, 13],
        'unit_price': [25.50, 45.00, 30.75, 25.50, 55.20, 45.00, 38.90, 30.75, 25.50, 55.20,
                      38.90, 45.00, 25.50, 30.75, 55.20, 38.90, 25.50, 45.00, 30.75, 55.20],
        'region': ['North', 'South', 'East', 'West', 'North', 'South', 'East', 'West', 'North', 'South',
                  'East', 'West', 'North', 'South', 'East', 'West', 'North', 'South', 'East', 'West'],
        'customer_type': ['Premium', 'Standard', 'Premium', 'Standard', 'Premium', 'Standard', 'Premium', 'Standard',
                         'Premium', 'Standard', 'Premium', 'Standard', 'Premium', 'Standard', 'Premium', 'Standard',
                         'Premium', 'Standard', 'Premium', 'Standard']
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field operations** for transformation

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_operations": {
            "lp_category": {  # Add new field
                "operation_type": "add_from_lookup",
                "base_on_column": "location_province",
                "lookup_table_name": "location_province_categories"
            }
        },
        "lookup_tables": {
            "location_province_categories": {
                "BC": "British Columbia",
                "SK": "Saskatchewan"
            }
        }
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_operations": {
            # Add new field: total_salary (calculated)
            "total_salary": {
                "operation_type": "add_conditional",
                "condition": (
                    "pd.notna(row['years_of_experience']) "
                    "and pd.notna(row['current_salary_cad']) "
                    "and row['years_of_experience'] > 0 "
                    "and row['current_salary_cad'] > 0"
                ),
                "value_if_true": "row['years_of_experience'] * row['current_salary_cad']",
                "value_if_false": 0
            },
            # Add new field: lp_category (from lookup)
            "lp_category": {
                "operation_type": "add_from_lookup",
                "base_on_column": "location_province",
                "lookup_table_name": "location_province_categories"
            },
        },
        "lookup_tables": {
            "location_province_categories": {
                "BC": "British Columbia",
                "SK": "Saskatchewan",
                "BE": "Belgium",
                "QC": "Quebec",
                "MB": "Manitoba"
            }
        }
    }

kwargs = create_config_kwargs()
field_operations = kwargs["field_operations"]
lookup_tables = kwargs["lookup_tables"]

# Validate base columns exist
print(f"\n🔍 Validating field configuration...\n")
for field_name, config in field_operations.items():
    base_column = config.get("base_on_column")
    if base_column and base_column not in df.columns:
        raise ValueError(
            f"❌ Base column '{base_column}' for field '{field_name}' not found in dataset!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update field_operations in create_config_kwargs()"
        )

print(f"✓ Fields to add/modify: {', '.join(field_operations.keys())}")
for field, config in field_operations.items():
    print(f"  {field}: {config['operation_type']}")

print(f"\n✓ Lookup tables: {', '.join(lookup_tables.keys())}")
for table_name, table_data in lookup_tables.items():
    print(f"  {table_name}: {len(table_data)} mappings")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="field_transformation",
    description="Add/modify fields using lookups and conditions.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description="Processing field operations",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create AddOrModifyFieldsOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_operations`: Dictionary defining add/modify operations
- `lookup_tables`: Dictionary or file paths for lookups
- `mode='REPLACE'`: Modify original fields (or 'ENRICH' for new columns)
- Operation types:
  - `add_constant`: Add field with constant value
  - `add_from_lookup`: Add field from lookup table
  - `add_conditional`: Add field based on condition
  - `modify_constant`: Modify field with constant
  - `modify_from_lookup`: Modify field from lookup
  - `modify_conditional`: Modify field based on condition
  - `modify_expression`: Modify field using expression
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = AddOrModifyFieldsOperation(
    # Core parameters
    field_operations=field_operations,
    lookup_tables=lookup_tables,
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field operations:  {len(field_operations)}")
print(f"  Lookup tables:     {len(lookup_tables)}")
print(f"  Mode:              {operation.mode}")
print(f"  Visualizations:    {operation.generate_visualization}")
print(f"  Save output:       {operation.save_output}")
print(f"  Force recalc:      {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing add/modify fields operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the processed output from task_dir
- Compare with original data
- Review metrics and artifacts

**Generated artifacts:**
- Processed CSV in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records with new fields
    print("\n🔍 Sample Processed Records (with new fields):")
    new_fields = [col for col in result_df.columns if col not in df.columns]
    display_cols = new_fields[:3]
    print(result_df[display_cols].head(10))
    
    # Field comparison
    print(f"\n📈 Field Changes:")
    original_fields = set(df.columns)
    result_fields = set(result_df.columns)
    added_fields = result_fields - original_fields
    modified_fields = [col for col in original_fields & result_fields 
                      if not df[col].equals(result_df[col])]
    
    print(f"  Original fields:    {len(df.columns)}")
    print(f"  Result fields:      {len(result_df.columns)}")
    print(f"  Added fields:       {len(added_fields)}")
    if added_fields:
        for field in sorted(added_fields):
            print(f"    • {field}")
    print(f"  Modified fields:    {len(modified_fields)}")
    if modified_fields:
        for field in sorted(modified_fields):
            print(f"    • {field}")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 New fields added for enhanced data analysis!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Processed CSV
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and effectiveness metrics
2. **Output Data**: Processed records with sample rows
3. **Visualizations**: Charts and graphs from the latest operation run

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / "metrics"

# ---------- Helper ----------
def format_value(v, float_precision=4):
    """Safely format metric values (handle NaN, floats, others)."""
    if isinstance(v, float):
        if math.isnan(v):
            return "N/A"
        return f"{v:.{float_precision}f}"
    return v


# ---------- Locate metrics ----------
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")

else:
    metrics_files = list(metrics_dir.glob("*.json"))

    if not metrics_files:
        print("⚠️  No metrics files found")

    else:
        # 1) Exclude data_types files
        filtered_files = [
            f for f in metrics_files
            if not f.name.startswith("data_types")
        ]

        if filtered_files:
            target_files = filtered_files
            print(f"✓ Found {len(filtered_files)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest file
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB")
        print("=" * 80)

        try:
            with open(latest_metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # ---------- METADATA ----------
            print("\n📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name     : {metadata.get('name', 'N/A')}")

            if "operation" in metadata:
                op = metadata["operation"]
                print(
                    f"   Operation: "
                    f"{op.get('class', 'N/A')}."
                    f"{op.get('function', 'N/A')}"
                )

            # ---------- FIELD CHANGES ----------
            print("\n📊 FIELD CHANGES:")
            print(f"   Fields Added   : {metrics.get('fields_added_count', 0)}")
            print(f"   Fields Modified: {metrics.get('fields_modified_count', 0)}")

            # ---------- SUMMARY STATISTICS ----------
            if "summary_statistics" in metrics:
                print("\n📈 SUMMARY STATISTICS:")

                for field, stats in metrics["summary_statistics"].items():
                    print(f"\n   {field}:")

                    # Numeric distribution
                    if "mean" in stats:
                        numeric_keys = [
                            "count", "mean", "std", "min",
                            "25%", "50%", "75%", "max"
                        ]
                        for k in numeric_keys:
                            if k in stats:
                                print(f"      {k:>6}: {format_value(stats[k])}")

                    # Categorical distribution
                    else:
                        cat_keys = ["count", "unique", "top", "freq"]
                        for k in cat_keys:
                            if k in stats:
                                print(f"      {k:>6}: {stats[k]}")

            # ---------- CORRELATIONS ----------
            print("\n🔗 CORRELATIONS (Original vs Modified):")
            correlations = metrics.get("correlations", {})

            if not correlations:
                print("   (No correlations available)")
            else:
                for field, corr in correlations.items():
                    print(f"   {field:30s}: {format_value(corr)}")

            # ---------- MISSING VALUES ----------
            if "missing_values" in metrics:
                print("\n❓ MISSING VALUES (New Fields):")
                for field, count in metrics["missing_values"].items():
                    print(f"   {field:30s}: {count}")

            # ---------- PERFORMANCE ----------
            print("\n⚡ PERFORMANCE:")
            print(f"   Execution Time   : {metrics.get('execution_time_seconds', 0):.4f}s")
            print(f"   Records Processed: {metrics.get('records_processed', 0):,}")

            rps = metrics.get("records_per_second", 0)
            if rps > 0:
                print(f"   Records/Second   : {rps:.2f}")
            else:
                print("   Records/Second   : N/A")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")

        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show added/modified fields comparison
            print("\n\n📊 Field Comparison:")
            print("-" * 80)
            original_cols = set(df.columns)
            output_cols = set(output_df.columns)
            added = output_cols - original_cols
            
            if added:
                print(f"\n✨ Added Fields ({len(added)}):")
                for col in sorted(added):
                    print(f"   • {col}")
                    # Show sample values
                    sample_vals = output_df[col].dropna().head(3).tolist()
                    print(f"      Sample values: {sample_vals}")
            
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure add/modify fields with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Add new fields using constants, lookups, or conditions
- Modify existing fields with expressions or conditional logic
- Lookup tables enable data enrichment from external sources
- Conditional operations allow complex business logic
- Comprehensive metrics track field changes and correlations

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_add_modify_fields_advanced.ipynb`** includes:
- Complex conditional expressions
- Multiple lookup table chaining
- Custom transformation functions
- Batch field operations
- Integration with external data sources
- Performance optimization for large datasets

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)